In [1]:
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn
import torchmetrics as tm

In [2]:
root_path = "/home/stefan/ioai-prep/kits/cls_ai_art"
device = "cuda" if torch.cuda.is_available() else "cpu"

seed = 42
torch.random.manual_seed(seed)

batch_size = 64

# Data

In [10]:
class ImageDataset(Dataset):
    def __init__(self, root_path: str, df: pd.DataFrame, transform=None):
        self.root_path = Path(root_path)
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = self.root_path / row["ImagePath"]
        image = Image.open(img_path)
        if image.mode == "P":
            image = image.convert("RGBA")
        image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        if "label" not in row:
            return image
        
        label = torch.tensor(row["Label"], dtype=torch.long)
        return image, label

In [11]:
train_df = pd.read_csv(f"{root_path}/train.csv")
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=seed)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ImageDataset(f"{root_path}", train_df, train_transform)
val_ds = ImageDataset(f"{root_path}", val_df, val_transform)
test_ds = ImageDataset(f"{root_path}", pd.read_csv(f"{root_path}/test.csv"), val_transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4)

test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4)

In [5]:
# sanity check
x = next(iter(train_loader))
[y.shape for y in x]

[torch.Size([64, 3, 224, 224]), torch.Size([64])]

# Model

In [6]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)

model(x[0].to(device)).shape

torch.Size([64, 2])

# Training

In [7]:
epochs = 5
lr = 1e-4

losses = []

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
f1_score = tm.F1Score(task='binary')

In [8]:
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        # forward pass
        logits = model(images)
        loss = criterion(logits, labels)

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # stats for nerds
        losses.append(loss.item())
        running_loss += loss.item()

    running_loss /= len(train_loader)

    # validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    val_loss /= len(val_loader)
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    f1 = f1_score(all_preds, all_labels)

    print(
        f"epoch {epoch}, loss={running_loss:.4f}, val_loss={val_loss:.4f}, f1={f1:.4f}"
    )

100%|██████████| 10/10 [00:05<00:00,  1.99it/s]


epoch 1, loss=0.6436, val_loss=0.6121, f1=0.7453


100%|██████████| 10/10 [00:05<00:00,  1.88it/s]


epoch 2, loss=0.4658, val_loss=0.5510, f1=0.7653


100%|██████████| 10/10 [00:05<00:00,  1.98it/s]


epoch 3, loss=0.3154, val_loss=0.4220, f1=0.8295


100%|██████████| 10/10 [00:04<00:00,  2.20it/s]


epoch 4, loss=0.2017, val_loss=0.3744, f1=0.8506


100%|██████████| 10/10 [00:05<00:00,  1.91it/s]


epoch 5, loss=0.1158, val_loss=0.3509, f1=0.8500


# Submission

In [21]:
model.eval()
all_predictions = []
	
with torch.no_grad():
    for images in tqdm(test_loader):
        images = images.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_predictions.extend(preds)
	
submission_df = test_ds.df.copy()
submission_df["Label"] = all_predictions
submission_df.drop(["ImagePath"], axis=1, inplace=True)
submission_df.to_csv(f"{root_path}/submission.csv", index=False)

100%|██████████| 4/4 [00:01<00:00,  2.02it/s]


In [22]:
submission_df.head()

,SampleID,Label
0,775,0
1,776,0
2,777,0
3,778,0
4,779,1
